<a href="https://colab.research.google.com/github/crowell97/ES2245/blob/main/es2245_lecture13.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EARTHSC 2245: Lecture 12 - Multiple Comparisons after ANOVA
**Introductory Data Analysis for Earth and Environmental Sciences (Chapter 11)**

This notebook explores the mathematical framework of Analysis of Variance (ANOVA), its inherent limitations in identifying specific group differences, and the subsequent statistical tests (Post-Hoc and A Priori) used to pinpoint where those variations occur.

## 1. ANOVA Basics and Limitations



ANOVA evaluates whether there is a statistically significant difference among the means of three or more independent groups by comparing variances. It breaks down the total variation in the data into two components:
* **Between-group Variation ($SS_{between}$):** The variation between the individual group means and the overall Grand Mean.
* **Within-group Variation ($SS_{within}$):** The natural error or random variation within each individual group.

The test calculates an **F-Ratio** by dividing the among-group variance by the within-group variance. A large treatment effect (significant difference between groups) or a very low within-group "noise" leads to a large F-value, pushing it into the rejection region of the F-distribution.



### The Primary Limitation of ANOVA
A significant one-way ANOVA result only tells us that *at least one* group mean is significantly different from the others. It operates as an "omnibus" test, meaning it does not specify *which* specific means differ.

To determine where the differences lie, we must consider the type of ANOVA model:
* **Model II (Random Effects):** Treatments are random representatives of a larger population. The primary goal is just to see if general variance exists, so further testing is usually unnecessary.
* **Model I (Fixed Effects):** Treatments are specifically chosen by the researcher. Identifying the exact differences between these specific treatments is crucial. Multiple comparison tests are designed precisely for Model I ANOVA.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import itertools

# Let's generate a synthetic dataset representing stable isotope values
# from three distinct geological sample sites.
np.random.seed(42)
site_a = np.random.normal(loc=10.5, scale=0.5, size=4)
site_b = np.random.normal(loc=10.8, scale=0.5, size=4)
site_c = np.random.normal(loc=13.2, scale=0.5, size=4)

# Assemble the data into a Pandas DataFrame
df = pd.DataFrame({
    'Isotope_Value': np.concatenate([site_a, site_b, site_c]),
    'Site': ['Site A']*4 + ['Site B']*4 + ['Site C']*4
})

# 1. Perform a One-Way ANOVA
f_stat, p_val = stats.f_oneway(site_a, site_b, site_c)
print(f"ANOVA F-statistic: {f_stat:.3f}")
print(f"ANOVA P-value: {p_val:.4f}")
print("-" * 40)
if p_val < 0.05:
    print("Conclusion: Reject the null hypothesis. At least one site's mean is different.")
    print("However, the ANOVA does not tell us whether A differs from B, B from C, or A from C.")

ANOVA F-statistic: 50.727
ANOVA P-value: 0.0000
----------------------------------------
Conclusion: Reject the null hypothesis. At least one site's mean is different.
However, the ANOVA does not tell us whether A differs from B, B from C, or A from C.


## 2. Post Hoc (A Posteriori) Tests



Post Hoc tests are conducted *after* a significant ANOVA result. Their primary purpose is to compare sets of means without artificially inflating the **Type 1 Error rate** (false positives). If a researcher indiscriminately runs multiple standard t-tests on every possible pair of groups, the probability of finding a "significant" result purely by chance increases drastically.

### Tukey's Honestly Significant Difference (HSD) Test
Tukey's HSD test is one of the most common post hoc procedures. It uses a stepwise approach to compare all possible pairs of means while strictly controlling the overall family-wise error rate.
* **Mechanism:** It orders the means from largest to smallest and systematically compares them.
* **Statistic:** It calculates a $q$ statistic, utilizing the Mean Square Error ($MS_{error}$) from the initial ANOVA as the most robust estimate of the overall population variance.
* **Trade-offs:** Because it is designed to strictly prevent false positives, Tukey's HSD is relatively conservative. In datasets with subtle differences, it can suffer from lower statistical power, sometimes resulting in ambiguous overlapping groups.

In [2]:
# 2. Perform Tukey's HSD Test
print("Tukey's HSD Test Results:")
print("-" * 40)

# We use an alpha level of 0.05 to maintain a 95% family-wise confidence level
tukey_result = pairwise_tukeyhsd(endog=df['Isotope_Value'], groups=df['Site'], alpha=0.05)
print(tukey_result)

# Interpretation:
# The 'reject' column indicates whether we reject the null hypothesis for that specific pair.
# Here, Site C is significantly different from both Site A and Site B.
# Site A and Site B are not significantly different from each other.

Tukey's HSD Test Results:
----------------------------------------
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
Site A Site B   0.2186 0.6675 -0.4778 0.9151  False
Site A Site C   2.2768    0.0  1.5804 2.9733   True
Site B Site C   2.0582    0.0  1.3618 2.7546   True
---------------------------------------------------


## 3. Alternative Multiple Comparison Approaches

Different research scenarios call for different levels of strictness when correcting for multiple comparisons:

* **Least Significant Difference (LSD):** Essentially a series of pairwise t-tests that substitute the standard pooled variance with the $MS_{error}$ from the ANOVA. It is the *least conservative* test; it offers high statistical power but comes with a severe risk of inflating Type 1 errors.
* **Bonferroni Correction:** A highly robust method that controls the error rate by dividing the target significance level ($\alpha$) by the total number of comparisons ($n$). It is extremely conservative and virtually guarantees no false positives, but often leads to a severe loss of statistical power (Type II errors).
* **Student-Newman-Keuls (SNK) Test:** A stepwise procedure similar to Tukey's, but it utilizes varying critical values depending on the specific "range" or distance between the ranked means being compared.
* **Scheffé's Test:** The most flexible and conservative post hoc test available. It is uniquely designed not just for pairwise comparisons, but for testing complex linear contrasts (e.g., comparing the average of Group A and B against Group C).

In [6]:
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Re-establish ANOVA parameters to ensure we have the global MS_error
model = smf.ols('Isotope_Value ~ Site', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=1)
ms_error = anova_table.loc['Residual', 'mean_sq']
df_error = anova_table.loc['Residual', 'df']

# Calculate means and counts, and sort them descending (Required for stepwise tests like SNK)
summary = df.groupby('Site')['Isotope_Value'].agg(['mean', 'count']).sort_values('mean', ascending=False)
sites = summary.index.tolist()
k = len(sites) # Number of groups
num_pairs = k * (k - 1) / 2 # Total number of pairwise comparisons
alpha = 0.05

print(f"Global MS_error: {ms_error:.4f} | Degrees of Freedom (Error): {df_error}")
print("=" * 65)

# Iterate through all unique pairs
for i in range(k):
    for j in range(i + 1, k):
        site1, site2 = sites[i], sites[j]
        mean1, mean2 = summary.loc[site1, 'mean'], summary.loc[site2, 'mean']
        n1, n2 = summary.loc[site1, 'count'], summary.loc[site2, 'count']

        print(f"Comparing {site1} vs {site2} (Means: {mean1:.2f} & {mean2:.2f})")

        # ---------------------------------------------------------
        # 1. Least Significant Difference (LSD)
        # Uses the global MS_error instead of local pooled variance
        se_diff = np.sqrt(ms_error * ((1/n1) + (1/n2)))
        t_stat = abs(mean1 - mean2) / se_diff
        p_lsd = 2 * (1 - stats.t.cdf(t_stat, df_error))
        sig_lsd = p_lsd < alpha
        print(f"  [LSD]        Significant? {str(sig_lsd):<5} (p-value: {p_lsd:.4f})")

        # ---------------------------------------------------------
        # 2. Student-Newman-Keuls (SNK)
        # Step size 'p' is the number of means in the range being compared (inclusive)
        step_size_p = j - i + 1
        # Calculate the q-statistic using the harmonic mean of sample sizes
        n_harmonic = 2 / ((1/n1) + (1/n2))
        q_stat = abs(mean1 - mean2) / np.sqrt(ms_error / n_harmonic)
        # Fetch the critical q-value from the studentized range distribution
        q_crit = stats.studentized_range.ppf(1 - alpha, k=step_size_p, df=df_error)
        sig_snk = q_stat > q_crit
        print(f"  [SNK]        Significant? {str(sig_snk):<5} (q-stat: {q_stat:.2f}, q-crit: {q_crit:.2f})")

        # ---------------------------------------------------------
        # 3. Bonferroni Correction
        # Multiply the LSD p-value by the total number of pairwise comparisons
        p_bonf = min(1.0, p_lsd * num_pairs)
        sig_bonf = p_bonf < alpha
        print(f"  [Bonferroni] Significant? {str(sig_bonf):<5} (Adjusted p-value: {p_bonf:.4f})")

        # ---------------------------------------------------------
        # 4. Scheffé's Test
        # Contrast F-statistic is the t-statistic squared
        f_stat = t_stat**2
        # Heavily penalized critical F-value
        f_crit = (k - 1) * stats.f.ppf(1 - alpha, k - 1, df_error)
        sig_scheffe = f_stat > f_crit
        print(f"  [Scheffé]    Significant? {str(sig_scheffe):<5} (F-stat: {f_stat:.2f}, F-crit: {f_crit:.2f})")
        print("-" * 65)

Global MS_error: 0.1244 | Degrees of Freedom (Error): 9.0
Comparing Site C vs Site B (Means: 13.09 & 11.03)
  [LSD]        Significant? True  (p-value: 0.0000)
  [SNK]        Significant? True  (q-stat: 11.67, q-crit: 3.20)
  [Bonferroni] Significant? True  (Adjusted p-value: 0.0001)
  [Scheffé]    Significant? True  (F-stat: 68.09, F-crit: 8.51)
-----------------------------------------------------------------
Comparing Site C vs Site A (Means: 13.09 & 10.82)
  [LSD]        Significant? True  (p-value: 0.0000)
  [SNK]        Significant? True  (q-stat: 12.91, q-crit: 3.95)
  [Bonferroni] Significant? True  (Adjusted p-value: 0.0000)
  [Scheffé]    Significant? True  (F-stat: 83.32, F-crit: 8.51)
-----------------------------------------------------------------
Comparing Site B vs Site A (Means: 11.03 & 10.82)
  [LSD]        Significant? False (p-value: 0.4035)
  [SNK]        Significant? False (q-stat: 1.24, q-crit: 3.20)
  [Bonferroni] Significant? False (Adjusted p-value: 1.0000)
  

## 4. Planned Comparisons (A Priori Tests)

Unlike post hoc tests, Planned Comparisons are formulated *before* the data is even collected, based on strong theoretical foundations, prior literature, or specific experimental designs (e.g., comparing a designated control group against several different experimental doses).

**Key Advantages:**
1. **Maximized Statistical Power:** Because the researcher is testing a few specific, predefined hypotheses rather than "fishing" through every possible combination, there is no need for the heavy, conservative mathematical penalties used in post hoc tests.
2. **Targeted Sensitivity:** Planned comparisons can often detect significant differences in specific areas of interest where conservative tests like Tukey's HSD or Bonferroni might fail.

Mathematically, these operate similarly to independent t-tests. However, instead of only using the variance from the two groups being compared, they utilize the $MS_{error}$ derived from the *entire* ANOVA model. This leverages the degrees of freedom from the whole dataset, providing a much more stable and accurate estimate of the true population variance.

In [5]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 4. Performing a Planned Comparison (A Priori)
# Hypothesis: Based on geographical positioning, we planned strictly to compare Site A and Site C.

# First, we need to extract the overall MS_error (Mean Square Error) from the ANOVA model.
# We use the formula API (smf.ols) so that statsmodels can easily generate the ANOVA table.
model = smf.ols('Isotope_Value ~ Site', data=df).fit()
anova_table = sm.stats.anova_lm(model, typ=1)

ms_error = anova_table.loc['Residual', 'mean_sq']
df_error = anova_table.loc['Residual', 'df']

# Extract specific group means and sample sizes
mean_a = df[df['Site'] == 'Site A']['Isotope_Value'].mean()
mean_c = df[df['Site'] == 'Site C']['Isotope_Value'].mean()
n_a = len(df[df['Site'] == 'Site A'])
n_c = len(df[df['Site'] == 'Site C'])

# Calculate the standard error of the difference using the global MS_error
se_diff = np.sqrt(ms_error * ((1/n_a) + (1/n_c)))

# Calculate the t-statistic for the planned comparison
t_planned = (mean_c - mean_a) / se_diff

# Calculate the two-tailed p-value using the global degrees of freedom
p_planned = 2 * (1 - stats.t.cdf(abs(t_planned), df_error))

print("Planned Comparison Results (Site C vs Site A):")
print("-" * 40)
print(f"Global MS_error (from full ANOVA): {ms_error:.4f}")
print(f"Global Degrees of Freedom (Error): {df_error}")
print(f"Calculated t-statistic: {t_planned:.3f}")
print(f"Calculated P-value: {p_planned:.4f}")

if p_planned < 0.05:
    print("\nConclusion: The means of Site A and Site C are significantly different.")
else:
    print("\nConclusion: There is no significant difference between the means of Site A and Site C.")

Planned Comparison Results (Site C vs Site A):
----------------------------------------
Global MS_error (from full ANOVA): 0.1244
Global Degrees of Freedom (Error): 9.0
Calculated t-statistic: 9.128
Calculated P-value: 0.0000

Conclusion: The means of Site A and Site C are significantly different.
